In [0]:
from pyspark.sql import functions as F

P = "olist_proyecto.plata"
O = "olist_proyecto.oro"

In [0]:
r = spark.table(f"{P}.orders").agg(F.min("fecha_compra").alias("ini"),
                                   F.max("fecha_compra").alias("fin")).first()

dim_fecha = (spark.sql(f"""
    SELECT explode(sequence(to_date('{r.ini.date()}'), to_date('{r.fin.date()}'),
                            interval 1 day)) AS fecha""")
  .withColumn("fecha_key", F.date_format("fecha", "yyyyMMdd").cast("int"))
  .withColumn("anio", F.year("fecha"))
  .withColumn("trimestre", F.quarter("fecha"))
  .withColumn("mes", F.month("fecha"))
  .withColumn("nombre_mes", F.date_format("fecha", "MMMM"))
  .withColumn("dia", F.dayofmonth("fecha"))
  .withColumn("dia_semana", F.date_format("fecha", "EEEE")))

dim_fecha.write.mode("overwrite").saveAsTable(f"{O}.dim_fecha")

In [0]:
dim_cliente = (spark.table(f"{P}.customers")
    .groupBy("customer_unique_id")
    .agg(F.first("customer_city").alias("ciudad"),
         F.first("customer_state").alias("estado"),
         F.first("customer_zip_code_prefix").alias("zip_prefix")))
dim_cliente.write.mode("overwrite").saveAsTable(f"{O}.dim_cliente")

dim_producto = (spark.table(f"{P}.products")
    .join(spark.table(f"{P}.category_translation"), "product_category_name", "left")
    .select("product_id",
            F.col("product_category_name").alias("categoria"),
            F.coalesce("product_category_name_english",
                       "product_category_name").alias("categoria_en"),
            F.col("product_weight_g").cast("double").alias("peso_g"),
            F.col("product_length_cm").cast("double").alias("largo_cm"),
            F.col("product_height_cm").cast("double").alias("alto_cm"),
            F.col("product_width_cm").cast("double").alias("ancho_cm")))
dim_producto.write.mode("overwrite").saveAsTable(f"{O}.dim_producto")

dim_vendedor = (spark.table(f"{P}.sellers")
    .select("seller_id",
            F.col("seller_city").alias("ciudad"),
            F.col("seller_state").alias("estado")))
dim_vendedor.write.mode("overwrite").saveAsTable(f"{O}.dim_vendedor")

In [0]:
# pedidos entregados, con el cliente real (customer_unique_id)
ped = (spark.table(f"{P}.orders")
    .filter("order_status = 'delivered'")
    .join(spark.table(f"{P}.customers").select("customer_id", "customer_unique_id"),
          "customer_id")
    .withColumn("fecha_key", F.date_format("fecha_compra", "yyyyMMdd").cast("int")))

base = ped.select("order_id", "customer_unique_id", "fecha_key")

fact_ventas = (spark.table(f"{P}.order_items").join(base, "order_id")
    .select("order_id", "order_item_id", "product_id", "seller_id",
            "customer_unique_id", "fecha_key",
            F.col("price").alias("precio"),
            F.col("freight_value").alias("flete"),
            (F.col("price") + F.col("freight_value")).alias("monto_facturado")))
fact_ventas.write.mode("overwrite").saveAsTable(f"{O}.fact_ventas")

fact_pagos = (spark.table(f"{P}.order_payments").join(base, "order_id")
    .select("order_id", "payment_sequential", "payment_type",
            F.col("payment_installments").alias("cuotas"),
            F.col("payment_value").alias("monto_cobrado"),
            "customer_unique_id", "fecha_key"))
fact_pagos.write.mode("overwrite").saveAsTable(f"{O}.fact_pagos")

rev = spark.table(f"{P}.order_reviews").select("order_id", "review_score")

fact_entregas = (ped.join(rev, "order_id", "left")
    .withColumn("dias_entrega",
        F.when(F.col("fecha_dudosa") == "no",
               F.datediff("fecha_entrega", "fecha_compra")))
    .withColumn("dias_retraso",
        F.when(F.col("fecha_dudosa") == "no",
               F.datediff("fecha_entrega", "fecha_estimada")))
    .select("order_id", "customer_unique_id", "fecha_key",
            "fecha_dudosa", "dias_entrega", "dias_retraso", "review_score"))
fact_entregas.write.mode("overwrite").saveAsTable(f"{O}.fact_entregas")

In [0]:
spark.sql("SHOW TABLES IN olist_proyecto.oro").show()

spark.sql(f"""
  SELECT (SELECT COUNT(*) FROM {P}.orders WHERE order_status='delivered') AS entregados,
         (SELECT COUNT(*) FROM {O}.fact_entregas) AS filas_fact_entregas,
         (SELECT COUNT(*) FROM {O}.fact_ventas)   AS filas_fact_ventas,
         (SELECT ROUND(SUM(monto_facturado),2) FROM {O}.fact_ventas) AS facturado,
         (SELECT ROUND(SUM(monto_cobrado),2)   FROM {O}.fact_pagos)  AS cobrado
""").show()